# Local Workshop: Ollama + RAG on 2024 Fish Species Discoveries

This notebook is designed for **local VS Code/Jupyter** execution and has four sections:
1. Setup Ollama locally
2. Intro to RAG with a 2024 fish-species source PDF
3. Advanced retrieval over fish-species content
4. Intent routing for Q&A vs summarization

## Local Prerequisites

Before running, make sure:
- Ollama is installed on your machine
- You can run `ollama` from a terminal
- You are using a Python environment with notebook dependencies installed

## How To Use This Notebook

Run the notebook from top to bottom in your local VS Code/Jupyter environment.

Expected local timing (varies by hardware):
- Setup and model pull: 3 to 20 minutes
- Index build and first query: 1 to 8 minutes

If you restart your kernel, rerun all cells in order.

## 1) Setup Ollama Locally

### What This Section Does

In this section you will:
- Install required Python packages
- Verify local Ollama CLI availability
- Start Ollama if it is not already running
- Pull one local LLM and one embedding model
- Run a quick smoke test to confirm local inference works

After this section, you should be able to run LLM calls without any external API key.

### 1.1 Install Python Dependencies

These packages provide the RAG framework and Ollama integrations.
Run once per fresh Colab runtime.

In [1]:
# Install Python dependencies for LlamaIndex + Ollama integrations
%pip install -q jedi pypdf pymupdf llama-index llama-index-llms-ollama llama-index-embeddings-ollama

Note: you may need to restart the kernel to use updated packages.


### 1.2 Verify Local Ollama Runtime

Check that the Ollama CLI is available on your machine.

In [2]:
# Verify Ollama CLI is available (PATH or common macOS locations)
import os
import shutil

candidates = [
    shutil.which("ollama"),
    "/opt/homebrew/bin/ollama",
    "/usr/local/bin/ollama",
    os.path.expanduser("~/.ollama/bin/ollama"),
]

OLLAMA_CMD = next((p for p in candidates if p and os.path.exists(p)), None)

if OLLAMA_CMD is None:
    raise RuntimeError(
        "Ollama CLI not found. Install Ollama from https://ollama.com/download "
        "or run 'brew install ollama', then restart VS Code/Jupyter and rerun."
    )

print(f"Ollama CLI found: {OLLAMA_CMD}")

Ollama CLI found: /opt/homebrew/bin/ollama


### 1.3 Start The Local Ollama Service

This checks whether Ollama is reachable at `127.0.0.1:11434` and starts it if needed.

In [3]:
# Start Ollama server if needed
import os
import subprocess
import time
import urllib.request

os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11434"

def ollama_api_up(url="http://127.0.0.1:11434/api/tags") -> bool:
    try:
        urllib.request.urlopen(url, timeout=2).close()
        return True
    except Exception:
        return False

if not ollama_api_up():
    subprocess.Popen([OLLAMA_CMD, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)

if not ollama_api_up():
    raise RuntimeError("Ollama server is not reachable at http://127.0.0.1:11434")

print("Ollama server is running.")
!ollama list

Ollama server is running.
]11;?\NAME                       ID              SIZE      MODIFIED       
nomic-embed-text:latest    0a109f422b47    274 MB    28 minutes ago    
llama3.2:3b                a80c4f17acd5    2.0 GB    29 minutes ago    


### 1.4 Pull Models (First Run Only)

We pull two models:
- `llama3.2:3b` for generation
- `nomic-embed-text` for embeddings

This keeps the full pipeline local with no API keys.

In [4]:
# Pull local models (safe to rerun)
LLM_MODEL = "llama3.2:3b"
EMBED_MODEL = "nomic-embed-text"

!ollama pull {LLM_MODEL}
!ollama pull {EMBED_MODEL}

print(f"Using LLM model: {LLM_MODEL}")
print(f"Using embedding model: {EMBED_MODEL}")

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling 970aa74c0a90: 100% ▕██████████████████▏ 274 MB                         
pulling c71d239df917: 100% ▕██████████████████▏  11 KB                         
pulling ce4a164fc046: 100% ▕

In [5]:
# Quick smoke test
!ollama run {LLM_MODEL} "Hello! Anyone home?"

]11;?\⠙ ⠙ ⠹ ⠼ ⠼ ⠴ ⠧ ⠧ ⠇ ⠏ ⠋ ⠙ Hello! I'm here and ready to chat. However, I don't have a physical presenc
presence or a personal space, so I don't have a "home" in the classical sen
sense. I exist solely as a digital entity, running on computer servers and 
responding to text-based inputs.

That being said, I'm always happy to converse with you! How can I assist yo
you today?



### Setup Checkpoint

If the smoke test responded correctly, your local model is ready.

If not:
- Re-run the server start cell
- Re-run the model pull cell
- Confirm `ollama list` shows `llama3.2:3b` and `nomic-embed-text`
- If memory errors continue, restart runtime and switch to `llama3.2:1b`

## 2) Intro to RAG

### RAG Concept In One Minute

RAG has two core steps:
1. Retrieval: find relevant chunks from your documents
2. Generation: use those chunks to produce an answer

This reduces hallucination by grounding answers in retrieved source text.

### 2.1 Configure LLM And Embedding Model

Here we configure the two core components of RAG:
- an LLM to generate answers
- an embedding model to map document chunks into vector space

In [6]:
# Configure LlamaIndex to use local Ollama models
from llama_index.core import Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

LLM_MODEL = globals().get("LLM_MODEL", "llama3.2:3b")
EMBED_MODEL = globals().get("EMBED_MODEL", "nomic-embed-text")

Settings.llm = Ollama(model=LLM_MODEL, request_timeout=180.0)
Settings.embed_model = OllamaEmbedding(model_name=EMBED_MODEL)
Settings.chunk_size = 512
Settings.chunk_overlap = 50

print(f"LLM configured: {LLM_MODEL}")
print(f"Embedding model configured: {EMBED_MODEL}")

LLM configured: llama3.2:3b
Embedding model configured: nomic-embed-text


### 2.2 Load 2024 Discovery Source Pages

This step downloads Natural History Museum **2024 discovery pages** and converts them into plain text files for indexing.
It includes the *Myloplus sauron* story and the broader 2024 new-species roundup.

In [19]:
import re
from pathlib import Path

import requests
from bs4 import BeautifulSoup

url = "https://www.nhm.ac.uk/discover/news/2024/december/dicaprios-snake-saurons-piranha-natural-history-museum-describe-190-new-species-2024.html"
out_path = Path("data/new_species_2024.txt")
out_path.parent.mkdir(parents=True, exist_ok=True)

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Referer": "https://www.nhm.ac.uk/",
}

response = requests.get(url, headers=headers, timeout=20)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

# NHM news pages expose article text in repeated "_text_*" blocks.
blocks = soup.find_all("div", class_=lambda c: isinstance(c, str) and c.startswith("_text_"))

def is_noise(text: str) -> bool:
    t = text.strip()
    if not t:
        return True
    if len(t) < 120:
        return True
    if re.fullmatch(r"\d{1,2}\s+[A-Za-z]+\s+\d{4}", t):
        return True
    noise_fragments = [
        "Receive email updates",
        "We use cookies",
        "best online experience",
        "Wildlife Photographer of the Year",
    ]
    return any(frag in t for frag in noise_fragments)

clean_paragraphs = []
for block in blocks:
    text = block.get_text(" ", strip=True)
    text = re.sub(r"\s+", " ", text).strip()
    if is_noise(text):
        continue
    clean_paragraphs.append(text)

# Fallback if page structure changes.
if not clean_paragraphs:
    content = soup.find("main") or soup
    text = re.sub(r"\s+", " ", content.get_text(" ", strip=True)).strip()
    clean_paragraphs = [text]

final_text = "\n\n".join(clean_paragraphs)
with out_path.open("w", encoding="utf-8") as f:
    f.write(f"SOURCE_URL: {url}\n\n")
    f.write(final_text)

source_files = [str(out_path)]
print(f"Saved cleaned source text to {out_path} ({len(final_text)} chars)")
print("Prepared source files:")
for path in source_files:
    print("-", path)

Saved cleaned source text to data/new_species_2024.txt (6818 chars)
Prepared source files:
- data/new_species_2024.txt


### 2.3 Build The Vector Index (Embedding Step)

During indexing, the embedding model converts the NHM document chunks into vectors.
These vectors are what retrieval uses to find relevant context.

In [13]:
# Build a basic RAG index from NHM 2024 source files
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

documents = SimpleDirectoryReader(input_files=source_files).load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine(similarity_top_k=4)

print(f"Loaded {len(documents)} document(s) and built index.")

Loaded 1 document(s) and built index.


### 2.4 LLM Output Without Retrieval (After Index Build)

This step still calls the Ollama LLM directly with **no retrieval step**.
Compare this with the RAG answer in the next cell to see grounding improvements.

In [14]:
# Direct LLM generation (no retrieval, no vector index lookup)
plain_prompt = "Name me a vegetarian piranha"
plain_response = Settings.llm.complete(plain_prompt)

# LlamaIndex LLM wrappers may return either a string-like object or an object with .text
print(getattr(plain_response, "text", str(plain_response)))

That's an interesting request!

One fictional example of a vegetarian piranha is "Piranha P", from the 1999 film "Piranha 3". In this movie, Piranha P is a genetically modified piranha that has been engineered to be vegetarian. This unique trait sets it apart from its carnivorous piranha cousins.


### 2.5 Run Diverse RAG Questions

These example queries cover multiple 2024 species discoveries so you can better demonstrate retrieval breadth, not just one fact.

In [15]:
# Intro RAG: run a diverse new-species question set
sample_questions = [
    "Name me a vegetarian piranha",
    "Which species in the NHM 2024 roundup were named after famous figures, and where were they found?",
    "What made the Carmenta brachyclados discovery in South Wales so unusual?",
    "Summarize the range of organism groups in the 2024 roundup (for example moths, beetles, copepods, amphipods) and why this diversity matters.",
    "How does the article connect new-species discovery to conservation and infrastructure decisions?",
]

for i, q in enumerate(sample_questions, 1):
    print(f"\n=== Question {i} ===")
    print(q)
    response = query_engine.query(q)
    print(response)


=== Question 1 ===
Name me a vegetarian piranha
There is no known species of piranha that is vegetarian. Piranhas are carnivorous fish, and they primarily feed on other small fish, crustaceans, and even carrion in some cases.

=== Question 2 ===
Which species in the NHM 2024 roundup were named after famous figures, and where were they found?
According to reports from recent natural history museum news, some of the newly discovered species have been named after notable individuals. The names include DiCaprio's snake, Sauron's piranha, among others that are mentioned as having names inspired by famous figures. However, specific information about their habitats is not provided in this context.

=== Question 3 ===
What made the Carmenta brachyclados discovery in South Wales so unusual?
The discovery was notable due to its fossilized remains being exceptionally well-preserved.

=== Question 4 ===
Summarize the range of organism groups in the 2024 roundup (for example moths, beetles, copepo

KeyboardInterrupt: 

## 3) Advanced Retrieval

### Why Advanced Retrieval Matters

A basic retriever may return partially relevant chunks.
Reranking adds an extra scoring step to keep the most relevant context before final answer generation.

In practice, this often improves factual precision and relevance.

### 3.1 Baseline Retrieval

Start with standard vector retrieval so you can compare quality before adding reranking.

In [ ]:
# Baseline retrieval
query = "Identify two unusual discovery stories in the NHM 2024 roundup and explain what each suggests about biodiversity monitoring and conservation."
baseline_engine = index.as_query_engine(similarity_top_k=6)
baseline_response = baseline_engine.query(query)

print("=== Baseline response ===")
print(baseline_response)

=== Baseline response ===
Two unusual discovery stories from the NHM 2024 roundup are the description of Carmenta brachyclados, a moth native to Guyana, found fluttering around a living room in south Wales, and DiCaprio's snake, Anguiculus dicaprioi.

The story of Carmenta brachyclados suggests that biodiversity monitoring needs to be more comprehensive, especially in unexpected places. The fact that this species was discovered in a living room, far from its native habitat, highlights the importance of continued exploration and discovery of new species, even in human-dominated environments. This discovery also underscores the need for accurate identification and classification of species, as initially misidentified specimens can lead to misconceptions about their distribution and ecology.

The discovery of DiCaprio's snake, Anguiculus dicaprioi, suggests that biodiversity monitoring needs to focus not only on the natural world but also on the unintended consequences of human activities

### 3.2 Add LLM Reranking

Reranking reviews the initially retrieved chunks and keeps only the most relevant ones for answer synthesis.

In [ ]:
# Advanced retrieval: LLM-based reranking
from llama_index.core.postprocessor import LLMRerank

reranker = LLMRerank(choice_batch_size=4, top_n=3)
rerank_engine = index.as_query_engine(
    similarity_top_k=10,
    node_postprocessors=[reranker],
)
rerank_response = rerank_engine.query(query)

print("=== Reranked response ===")
print(rerank_response)

=== Reranked response ===
Two unusual discovery stories in the NHM 2024 roundup are the naming of DiCaprio's snake (Anguiculus dicaprioi) after actor Leonardo DiCaprio, and Sauron's piranha (Myloplus sauron).

DiCaprio's snake suggests that even the names we give to species can be a reflection of the interests and passions of humans. The fact that this snake was named after an environmentalist implies that the discovery of new species is not just about cataloging existing life, but also about recognizing the connections between conservation and human culture.

The naming of Sauron's piranha also highlights the importance of biodiversity monitoring in conservation efforts. By describing a species that comes from a region where a dam project has put its future at risk, scientists are drawing attention to the impact of human activities on the natural world. This suggests that conservation efforts should not only focus on preserving existing ecosystems but also on understanding and mitigat

In [ ]:
# Compare retrieved source chunks
print("--- Baseline source nodes ---")
for i, node in enumerate(baseline_response.source_nodes, 1):
    print(f"[{i}] score={node.score:.4f} | {node.text[:120]}...")

print()
print("--- Reranked source nodes ---")
for i, node in enumerate(rerank_response.source_nodes, 1):
    print(f"[{i}] score={node.score:.4f} | {node.text[:120]}...")

--- Baseline source nodes ---
[1] score=0.7769 | From dinosaurs that may have roamed like cattle along Britain’s ancient waterways to tiny diatoms ceaselessly producing ...
[2] score=0.7702 | “But since these projects have been launched, we’re finding that there’s probably upwards of 70 endemic species found in...
[3] score=0.7635 | “We have described this remarkable moth, which we know is from central Guyana, but the only specimens and other material...
[4] score=0.7611 | 30 December 2022 Science news Dinosaurs and meteorites: Museum scientists described 552 new species in 2021 Museum scien...
[5] score=0.7565 | But the biggest fossil find this year came from the Isle of Wight, as scientists named a new dinosaur from the island . ...
[6] score=0.7412 | SOURCE_URL: https://www.nhm.ac.uk/discover/news/2024/december/dicaprios-snake-saurons-piranha-natural-history-museum-des...

--- Reranked source nodes ---
[1] score=10.0000 | SOURCE_URL: https://www.nhm.ac.uk/discover/news/2024/december

In [ ]:
# --- Experiment with your own parameters ---
EXP_CHUNK_SIZE = 256   # try 128, 256, 512, 1024
EXP_CHUNK_OVERLAP = 25 # try 0, 25, 50, 100
EXP_TOP_N = 3          # number of chunks kept after reranking

Settings.chunk_size = EXP_CHUNK_SIZE
Settings.chunk_overlap = EXP_CHUNK_OVERLAP

# Reuse already-clean extracted documents from Section 2.3
exp_index = VectorStoreIndex.from_documents(documents)
exp_reranker = LLMRerank(choice_batch_size=5, top_n=EXP_TOP_N)
exp_engine = exp_index.as_query_engine(
    similarity_top_k=10,
    node_postprocessors=[exp_reranker],
)

exp_response = exp_engine.query(query)
print(f"chunk_size={EXP_CHUNK_SIZE}, chunk_overlap={EXP_CHUNK_OVERLAP}, top_n={EXP_TOP_N}")
print(exp_response)

chunk_size=256, chunk_overlap=25, top_n=3
The two unusual discovery stories that stand out are the clearwing moth and the bryozoans from Australia. 

The story of the clearwing moth highlights the importance of collecting data from various sources, including social media platforms like Instagram. This discovery emphasizes the value of crowd-sourced information in biodiversity monitoring. The finding also underscores the need for accurate identification of species to avoid misclassification or overlooking of new species.

The bryozoans from Australia suggest that there is still much to be discovered in our planet's ancient history, providing a wealth of new data on past ecosystems and responses to environmental changes. This find highlights the importance of continued exploration and research into fossil records to better understand the evolution of biodiversity.


### Interpreting The Source Nodes

When comparing baseline vs reranked source nodes, look for:
- Higher relevance to the query intent
- Less off-topic context
- Better support for key claims in the final answer

## 4) Intent Routing: Q&A vs Summarize

In this section, we add a lightweight **router** in front of the RAG pipeline.
The router inspects the user request and selects one of two paths:

- `q&a`: Retrieve evidence and answer a new-species question directly.
- `summarize`: Retrieve broader evidence and generate a concise new-species summary.

### Why This Matters

Routing is useful when one assistant needs to support multiple interaction styles without forcing the user to choose manually.
It also mirrors production agent patterns where a classifier or policy decides which tool/chain to run.

In this demo, routing uses different retrieval settings for each task:
- Q&A path uses tighter context (`similarity_top_k=4`) for precise answers
- Summarize path uses broader context (`similarity_top_k=10`) for better coverage

This lets you see why one generic chain can look okay, but still underperform for one of the two intents.

### How This Demo Router Works

1. Read the user prompt.
2. Detect summary intent from keywords such as `summarize`, `summary`, `overview`, `brief`, or `key points`.
3. If summary intent is detected, run a summarization prompt over retrieved context.
4. Otherwise, run normal reranked RAG Q&A.

The first demo question now uses a cross-species summary ask:
- `Summarize the most unusual 2024 new-species stories and why naming species matters for conservation.`

In [ ]:
# Section 4 router utilities
def route_intent(user_text: str) -> str:
    text = user_text.lower()
    summarize_keywords = ["summarize", "summary", "overview", "brief", "key points"]
    if any(k in text for k in summarize_keywords):
        return "summarize"
    return "q&a"

def run_summary_path(user_text: str, top_k: int = 10):
    retriever = index.as_retriever(similarity_top_k=top_k)
    nodes = retriever.retrieve(user_text)
    context = "\n\n".join([n.node.get_content() for n in nodes])
    prompt = (
        "Summarize the following retrieved context in 5-8 bullet points. "
        "Focus on species names, unusual discovery stories, habitats, and conservation implications.\n\n"
        f"Context:\n{context}"
    )
    result = Settings.llm.complete(prompt)
    return getattr(result, "text", str(result)), nodes

def run_qa_path(user_text: str, top_k: int = 4):
    qa_engine = index.as_query_engine(similarity_top_k=top_k, node_postprocessors=[reranker])
    response = qa_engine.query(user_text)
    return str(response), response.source_nodes

def run_routed_query(user_text: str):
    route = route_intent(user_text)
    if route == "summarize":
        answer, nodes = run_summary_path(user_text, top_k=10)
    else:
        answer, nodes = run_qa_path(user_text, top_k=4)
    return route, answer, nodes

def run_forced_one_size_fits_all(user_text: str):
    answer, nodes = run_qa_path(user_text, top_k=4)
    return "q&a", answer, nodes

def print_comparison(user_text: str, preview_chars: int = 900):
    routed_route, routed_answer, routed_nodes = run_routed_query(user_text)
    forced_route, forced_answer, forced_nodes = run_forced_one_size_fits_all(user_text)

    print("\n=== Query ===")
    print(user_text)
    print(f"Routed selected: {routed_route} | retrieved nodes: {len(routed_nodes)}")
    print(f"Forced single-path route: {forced_route} | retrieved nodes: {len(forced_nodes)}")

    print("--- Routed answer preview ---")
    print(routed_answer[:preview_chars])

    print("--- Forced single-path answer preview ---")
    print(forced_answer[:preview_chars])

### 4.1 Run The Built-In Comparison Examples

This cell runs two prompts and compares:
- Routed behavior (intent-aware path)
- Forced single-path behavior (always Q&A settings)

The first example is now a multi-entity summary request:
- `Summarize the most unusual 2024 new-species stories and why naming species matters for conservation.`

In [ ]:
# Compare routed behavior vs forced single-path behavior
demo_queries = [
    "Summarize the most unusual 2024 new-species stories and why naming species matters for conservation.",
    "Where was Carmenta brachyclados discovered, where is it native to, and why is that unusual?",
]

for q in demo_queries:
    print_comparison(q)


=== Query ===
Summarize the most unusual 2024 new-species stories and why naming species matters for conservation.
Routed selected: summarize | retrieved nodes: 6
Forced single-path route: q&a | retrieved nodes: 2
--- Routed answer preview ---
Here are 5-8 bullet points summarizing the context, focusing on species names, unusual discovery stories, habitats, and conservation implications:

• The Natural History Museum described 190 new species in 2024, including 351 new species in 2022, 552 new species in 2021, and 503 new species in 2020.

• Some of the newly discovered species include:
 + Deroplatys xuzhengfai, a preying mantis from Borneo
 + Microhoria melecisi, an ant-mimic beetle from Crete
 + Kamchatkapora ozhgibesovi, a living bryozoan from the Sea of Okhotsk off Russia
 + Whizzing over to the Atlantic Ocean, two new species of pirate spiders were described from St Helena
 + A new dinosaur, Comptonatus chasei, discovered in the Isle of Wight
 + DiCaprio's snake, Anguiculus dicap

### 4.2 Try Your Own Prompt

Use this cell to test your own query. The router will choose a path automatically,
and you will still see the comparison against a forced one-size-fits-all path.

In [ ]:
# Interactive router demo
custom_query = input("Ask a new-species question or request a summary: ").strip()
if custom_query:
    print_comparison(custom_query)
else:
    print("No query entered.")


=== Query ===
name me a vegetarian piranha
Routed selected: q&a | retrieved nodes: 0
Forced single-path route: q&a | retrieved nodes: 1
--- Routed answer preview ---
Empty Response
--- Forced single-path answer preview ---
Myloplus sauron.


### Run This Notebook In Colab Or VS Code

Yes, this demo is feasible in both environments.

Use Colab when you want minimal local setup and a classroom-friendly start.
Use VS Code on your own machine when you want persistence, repeatability, and faster iteration.

Local VS Code checklist:
- Install Ollama on your machine
- Ensure `ollama serve` is running
- Pull models once: `llama3.2:3b` and `nomic-embed-text`
- Use a Python environment with notebook dependencies installed

Colab checklist:
- Enable a GPU runtime (T4 is fine)
- Run setup cells to install and start Ollama inside the runtime
- Re-run setup after runtime resets

Tip: keep the same new-species prompts in both environments to compare speed and output consistency.

### Suggested Exercises

Try these quick experiments:
- Change `similarity_top_k` from 4 to 8 in intro RAG
- Change reranker `top_n` from 3 to 2 or 5
- Ask multi-part questions and observe whether reranking helps